# T09 — Query-Techniken

Die Baseline embeddet die Frage, wie sie ist. Weicht ihre Formulierung von den Dokumenten ab, landen relevante Chunks unter Umständen zu weit unten. Zwei Techniken ändern nur die Frage, ohne Vector Store oder LLM anzufassen:

| Technik | Idee |
|---|---|
| **Query Expansion** | N Umformulierungen erzeugen, für jede suchen, die Ranglisten mit RRF vereinen |
| **HyDE** | Das LLM eine hypothetische Antwort entwerfen lassen und damit als zusätzliche Frage suchen |

Alle drei Agenten unten beantworten dieselbe Frage; die letzte Zelle vergleicht die gefundenen Quelldateien.

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Setup

Zuerst T01 ausführen: Dort entsteht der Vector Store unter `backend/db/data_vs.db`. Ist der Store leer, baut `load_or_build_vector_store()` ihn aus allen Dokumenten auf; das dauert einige Minuten.

In [ ]:
import sys

from loguru import logger

from ki_kompass_rag.baseline_rag import (
    RETRIEVER_TOP_K,
    SYSTEM_PROMPT,
    ask,
    build_agent,
    build_llm,
    load_or_build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")

vector_store, embedding_model = await load_or_build_vector_store()
llm = build_llm("openai")

QUERY = "Which pallets in our portfolio have a third-party verified EPD?"

## Baseline

In [ ]:
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
)
baseline = await ask(agent, QUERY)

## Query Expansion

Das LLM erzeugt `N` Umformulierungen der Frage. Jede wird separat embedded und gesucht, und die Ranglisten werden mit Reciprocal Rank Fusion (RRF) vereint, wie beim hybriden Retrieval in T07.

```
Original query ──► embed ──► top-k list 0 ─┐
Rephrasing 1   ──► embed ──► top-k list 1 ─┼─► RRF ──► final top-k
...                                        │
Rephrasing N   ──► embed ──► top-k list N ─┘
```

Kosten: ein zusätzlicher LLM-Aufruf und N zusätzliche Embedding-Aufrufe pro Frage. Umformulierungen können von der ursprünglichen Absicht abweichen; das DEBUG-Log unten zeigt sie.

In [ ]:
expansion_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=5,
)

sink = logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.utils.retriever",
    format="{message}",
)
expansion = await ask(expansion_agent, QUERY)
logger.remove(sink)

## HyDE: Hypothetical Document Embeddings

Eine Frage und der Abschnitt, der sie beantwortet, sind unterschiedlich formuliert; ihre Embeddings liegen deshalb weniger nah beieinander als möglich. Bei HyDE entwirft das LLM ohne Kontext eine hypothetische Antwort. Sie darf falsch sein, solange sie sich wie die Dokumente liest. Der Entwurf wird **zusätzlich** zur Frage embedded, und beide Ranglisten werden mit RRF vereint.

```
Query ─────────────────────────────────────────► embed ──► top-k ─┐
Query ──► LLM (no context) ──► hypothetical answer ──► embed ──► top-k ─┴─► RRF
```

Kosten: ein zusätzlicher LLM-Aufruf und ein zusätzlicher Embedding-Aufruf pro Frage. Geht der Entwurf in die falsche Richtung, zieht er unpassende Chunks herein.

In [ ]:
hyde_agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
    enable_hyde=True,
)

sink = logger.add(
    sys.stderr,
    level="DEBUG",
    filter="conversational_toolkit.utils.retriever",
    format="{message}",
)
hyde = await ask(hyde_agent, QUERY)
logger.remove(sink)

## Vergleich

In [ ]:
for name, answer in [("Baseline", baseline), ("Expansion", expansion), ("HyDE", hyde)]:
    print(f"{name}: {sorted({s.metadata['source_file'] for s in answer.sources})}")